In [ ]:
from pyspark.sql import SparkSession
# коммент
spark = (
    SparkSession.builder
        .appName("Перенос данных из Psql в Ch")
        .config(
            "spark.jars.packages",
            "org.postgresql:postgresql:42.5.0"
        )
        .getOrCreate()
)

print("Активная Spark сессия:", spark.sparkContext.uiWebUrl)


In [ ]:
import os
# Подключение к psql

jdbc_url    = 'jdbc:postgresql://postgres_source:5432/source' # строка подключения к постгре
table_name  = 'public.regions'                                # название таблицы и схемы
db_user     = os.getenv('POSTGRES_USER')                      # имя пользователя
db_password = os.getenv('POSTGRES_PASSWORD')                  # пароль

shop_df = spark.read                          \
    .format('jdbc')                           \
    .option('url', jdbc_url)                  \
    .option('user', db_user)                  \
    .option('password', db_password)          \
    .option('dbtable', 'public.shops')       \
    .option('fetchsize', 1000)                \
    .option('driver', 'org.postgresql.Driver')\
    .load()

shop_tz_df = spark.read                       \
    .format('jdbc')                           \
    .option('url', jdbc_url)                  \
    .option('user', db_user)                  \
    .option('password', db_password)          \
    .option('dbtable', 'public.shop_timezone')\
    .option('fetchsize', 1000)                \
    .option('driver', 'org.postgresql.Driver')\
    .load()

shop_df.createOrReplaceTempView("shops")
shop_tz_df.createOrReplaceTempView("shop_timezone")



In [13]:
a = spark.sql ("""
      SELECT st_id, shop_name,
            CASE 
                WHEN time_zone = 'RUS04' THEN 4 
                WHEN time_zone = ''      THEN 3
                ELSE RIGHT(time_zone, 1) 
            END as tz_code
      FROM shops             AS s
          JOIN shop_timezone AS st ON st.plant = s.st_id
        
""")
a.show(100)



+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  842|      Lenta|      3|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
|  848|      Lenta|      3|
|  848|      Lenta|      3|
+-----+-----------+-------+

